# Pulpermill: cards and chapters

Run the cells top to bottom (the model takes a few minutes to load), then copy the URL printed by the last cell into the Pulpermill page. There are no datasets to update: every session starts by cloning the code from GitHub.

**Set up once**
- Notebook settings: Internet on, accelerator GPU T4 x2.
- Add-ons, Secrets: `NGROK_AUTHTOKEN` (your ngrok token, required) and `HF_TOKEN` (optional: the model repo is not gated).
- The page is `index.html` in the repo. Open it from your computer (double-click) or from the repo's GitHub Pages address, and paste the URL below into it.

**Changing the code:** push to the repo and start a new session. Re-running the clone cell in a session that already imported the code would keep running the old version.


In [ ]:
# Install
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q
!pip install pydantic huggingface_hub fastapi uvicorn pyngrok -q


In [ ]:
# Get the code: a fresh clone every session (sessions are not persistent)
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/LacChe/pulpermill"
REPO_REF = None                      # a branch, tag or commit to run instead of the default branch
REPO_DIR = "/kaggle/working/pulpermill"

shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR] if REPO_REF is None else ["git", "clone", REPO_URL, REPO_DIR], check=True)
if REPO_REF is not None:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_REF], check=True)
sys.path.insert(0, REPO_DIR)
print("running commit", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip())


In [ ]:
# Settings. The defaults are in pulpermill/config.py; change any of them here, before the next cells run.
from pulpermill import config

# config.override(N_CTX=8192, GRAMMAR_MODE="fallback")


In [ ]:
# Load the model (downloads it the first time in a session)
from pulpermill import model

model.load_model()


In [ ]:
# Start the server and the ngrok tunnel. Paste the printed URL into the HTML page.
import threading, time
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from pulpermill import config
from pulpermill.server import app            # imported after the settings cell: the request limits are built when this loads

ngrok.set_auth_token(UserSecretsClient().get_secret(config.NGROK_SECRET))
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=config.PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(1.5)
url = ngrok.connect(config.PORT).public_url
print("Paste this into the card generator page:\n\n   ", url)
